# Görüntü Sahteciliği Tespiti — ViT & Swin Transformer Eğitimi

**Veri seti:** CASIA v2 (authentic vs tampered)  
**Modeller:** `google/vit-base-patch16-224` + `microsoft/swin-tiny-patch4-window7-224`  
**Hedef:** İkili sınıflandırma (0 = authentic, 1 = tampered) + ONNX export

## Notebook İçeriği
1. Kurulum ve GPU kontrolü
2. Veri seti indirme (CASIA v2)
3. Veri ön işleme + train/val/test split
4. ViT eğitimi
5. Swin Transformer eğitimi
6. Karşılaştırma + confusion matrix
7. ONNX export (her iki model için)
8. ONNX inference testi

> Colab Pro kullanıyorsan **Runtime → Change runtime type → A100 GPU** seç.

## 1. Kurulum

In [ ]:
!pip install -q numpy==1.26.4
!pip install -q transformers==4.44.2 datasets==2.20.0 accelerate==0.33.0 evaluate scikit-learn
!pip install -q onnx onnxruntime onnxruntime-gpu optimum[exporters]
!pip install -q matplotlib seaborn pillow

# ÖNEMLI: Bu hücreden sonra Runtime > Restart session yap, sonra alttaki hücrelerden devam et.

In [ ]:
import torch
import os
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Veri Seti — CASIA v2

**İki yöntem var:**
- **A) Kaggle'dan indir** (önerilen) — `kaggle.json` API anahtarını yüklemen gerek
- **B) HuggingFace datasets** üzerinden

### Yöntem A: Kaggle
1. https://www.kaggle.com → Profile → Account → Create New API Token → `kaggle.json` indir  
2. Aşağıdaki hücreyi çalıştırıp dosyayı yükle

In [ ]:
from google.colab import files
import os, shutil

if not os.path.exists('/root/.kaggle/kaggle.json'):
    print("kaggle.json yükle:")
    uploaded = files.upload()
    os.makedirs('/root/.kaggle', exist_ok=True)
    shutil.move('kaggle.json', '/root/.kaggle/kaggle.json')
    os.chmod('/root/.kaggle/kaggle.json', 0o600)
    print('kaggle.json kuruldu.')
else:
    print('kaggle.json zaten var.')

In [ ]:
!pip install -q kaggle
!kaggle datasets download -d sophatvathana/casia-dataset -p /content/data --unzip

In [ ]:
import os
for root, dirs, files in os.walk('/content/data'):
    depth = root.count(os.sep) - '/content/data'.count(os.sep)
    if depth <= 2:
        print('  ' * depth + os.path.basename(root) + f'/  ({len(files)} files)')

## 3. Veri Ön İşleme

CASIA v2'de iki klasör var: `Au` (authentic) ve `Tp` (tampered). Bunları binary label'a çeviriyoruz.

In [ ]:
import glob
from pathlib import Path

DATA_ROOT = '/content/data/CASIA2'

authentic_paths = []
tampered_paths = []

for p in Path(DATA_ROOT).rglob('*'):
    if p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.tif', '.bmp'}:
        if 'Au' in p.parts:
            authentic_paths.append(str(p))
        elif 'Tp' in p.parts:
            tampered_paths.append(str(p))

print(f'Authentic: {len(authentic_paths)}')
print(f'Tampered:  {len(tampered_paths)}')

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.DataFrame({
    'image_path': authentic_paths + tampered_paths,
    'label': [0] * len(authentic_paths) + [1] * len(tampered_paths),
})
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

train_df, test_df = train_test_split(df, test_size=0.15, stratify=df['label'], random_state=42)
train_df, val_df = train_test_split(train_df, test_size=0.15, stratify=train_df['label'], random_state=42)

print(f'Train: {len(train_df)}  Val: {len(val_df)}  Test: {len(test_df)}')
print('\nClass dağılımı (train):')
print(train_df["label"].value_counts())

In [ ]:
from datasets import Dataset, Image as HFImage, DatasetDict

def to_hf(df):
    return Dataset.from_pandas(df.rename(columns={'image_path': 'image'}), preserve_index=False)\
                  .cast_column('image', HFImage())

ds = DatasetDict({
    'train': to_hf(train_df),
    'validation': to_hf(val_df),
    'test': to_hf(test_df),
})
ds

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for i, ax in enumerate(axes.flat):
    sample = ds['train'][i]
    ax.imshow(sample['image'])
    ax.set_title('Authentic' if sample['label'] == 0 else 'Tampered')
    ax.axis('off')
plt.tight_layout(); plt.show()

## 4. ViT Eğitimi

Pretrained `vit-base-patch16-224` üzerine fine-tune. 5 epoch genellikle yeterli.

In [ ]:
from transformers import AutoImageProcessor, AutoModelForImageClassification
from transformers import TrainingArguments, Trainer
import numpy as np
import evaluate

LABELS = ['authentic', 'tampered']
id2label = {i: l for i, l in enumerate(LABELS)}
label2id = {l: i for i, l in enumerate(LABELS)}

def build_transform(processor):
    def transform(batch):
        images = [img.convert('RGB') for img in batch['image']]
        out = processor(images=images, return_tensors='pt')
        return {'pixel_values': out['pixel_values'], 'labels': batch['label']}
    return transform

accuracy = evaluate.load('accuracy')
f1 = evaluate.load('f1')

def compute_metrics(eval_pred):
    preds = np.argmax(eval_pred.predictions, axis=1)
    return {
        'accuracy': accuracy.compute(predictions=preds, references=eval_pred.label_ids)['accuracy'],
        'f1': f1.compute(predictions=preds, references=eval_pred.label_ids)['f1'],
    }

In [ ]:
VIT_CKPT = 'google/vit-base-patch16-224'
vit_processor = AutoImageProcessor.from_pretrained(VIT_CKPT)
vit_model = AutoModelForImageClassification.from_pretrained(
    VIT_CKPT, num_labels=2, id2label=id2label, label2id=label2id,
    ignore_mismatched_sizes=True,
)

ds_vit = ds.with_transform(build_transform(vit_processor))

In [ ]:
vit_args = TrainingArguments(
    output_dir='/content/checkpoints/vit',
    num_train_epochs=5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    fp16=True,
    logging_steps=20,
    report_to='none',
    remove_unused_columns=False,
)

vit_trainer = Trainer(
    model=vit_model,
    args=vit_args,
    train_dataset=ds_vit['train'],
    eval_dataset=ds_vit['validation'],
    compute_metrics=compute_metrics,
)
vit_trainer.train()

In [ ]:
vit_test_results = vit_trainer.evaluate(ds_vit['test'])
print('ViT Test:', vit_test_results)

vit_trainer.save_model('/content/checkpoints/vit_best')
vit_processor.save_pretrained('/content/checkpoints/vit_best')

## 5. Swin Transformer Eğitimi

In [ ]:
SWIN_CKPT = 'microsoft/swin-tiny-patch4-window7-224'
swin_processor = AutoImageProcessor.from_pretrained(SWIN_CKPT)
swin_model = AutoModelForImageClassification.from_pretrained(
    SWIN_CKPT, num_labels=2, id2label=id2label, label2id=label2id,
    ignore_mismatched_sizes=True,
)
ds_swin = ds.with_transform(build_transform(swin_processor))

In [ ]:
swin_args = TrainingArguments(
    output_dir='/content/checkpoints/swin',
    num_train_epochs=5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    fp16=True,
    logging_steps=20,
    report_to='none',
    remove_unused_columns=False,
)

swin_trainer = Trainer(
    model=swin_model,
    args=swin_args,
    train_dataset=ds_swin['train'],
    eval_dataset=ds_swin['validation'],
    compute_metrics=compute_metrics,
)
swin_trainer.train()

In [ ]:
swin_test_results = swin_trainer.evaluate(ds_swin['test'])
print('Swin Test:', swin_test_results)

swin_trainer.save_model('/content/checkpoints/swin_best')
swin_processor.save_pretrained('/content/checkpoints/swin_best')

## 6. Karşılaştırma + Confusion Matrix

In [ ]:
import pandas as pd

comparison = pd.DataFrame({
    'Model':    ['ViT-Base/16', 'Swin-Tiny'],
    'Accuracy': [vit_test_results['eval_accuracy'], swin_test_results['eval_accuracy']],
    'F1':       [vit_test_results['eval_f1'],       swin_test_results['eval_f1']],
    'Loss':     [vit_test_results['eval_loss'],     swin_test_results['eval_loss']],
})
comparison

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt

def plot_cm(trainer, ds_test, title):
    preds = trainer.predict(ds_test)
    y_pred = preds.predictions.argmax(axis=1)
    y_true = preds.label_ids
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(4, 3.5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=LABELS, yticklabels=LABELS)
    plt.title(title); plt.ylabel('True'); plt.xlabel('Predicted')
    plt.show()
    print(classification_report(y_true, y_pred, target_names=LABELS))

plot_cm(vit_trainer,  ds_vit['test'],  'ViT — Confusion Matrix')
plot_cm(swin_trainer, ds_swin['test'], 'Swin — Confusion Matrix')

## 7. ONNX Export

Hugging Face'in `optimum` aracı tek satırla yapıyor. Hiçbir abartı yok.

In [ ]:
!optimum-cli export onnx --model /content/checkpoints/vit_best  --task image-classification /content/onnx/vit
!optimum-cli export onnx --model /content/checkpoints/swin_best --task image-classification /content/onnx/swin
!ls -lh /content/onnx/vit /content/onnx/swin

## 8. ONNX Inference Testi (Backend için sanity check)

In [ ]:
import onnxruntime as ort
import numpy as np
from PIL import Image

def run_onnx(onnx_dir, processor, image_path):
    session = ort.InferenceSession(f'{onnx_dir}/model.onnx', providers=['CPUExecutionProvider'])
    img = Image.open(image_path).convert('RGB')
    inputs = processor(images=img, return_tensors='np')
    logits = session.run(None, {'pixel_values': inputs['pixel_values'].astype(np.float32)})[0]
    probs = np.exp(logits) / np.exp(logits).sum(axis=1, keepdims=True)
    pred = int(probs.argmax(axis=1)[0])
    return LABELS[pred], float(probs[0, pred])

sample_img = test_df.iloc[0]['image_path']
true_label = LABELS[test_df.iloc[0]['label']]

vit_pred,  vit_conf  = run_onnx('/content/onnx/vit',  vit_processor,  sample_img)
swin_pred, swin_conf = run_onnx('/content/onnx/swin', swin_processor, sample_img)

print(f'Gerçek:      {true_label}')
print(f'ViT tahmin:  {vit_pred}  ({vit_conf:.3f})')
print(f'Swin tahmin: {swin_pred} ({swin_conf:.3f})')

## 9. Modelleri İndir

Backend'e ekleyeceğin dosyalar `vit.zip` ve `swin.zip`.

In [ ]:
import shutil
shutil.make_archive('/content/vit',  'zip', '/content/onnx/vit')
shutil.make_archive('/content/swin', 'zip', '/content/onnx/swin')

from google.colab import files
files.download('/content/vit.zip')
files.download('/content/swin.zip')

---
## Backend Tarafında Kullanım Özeti

İndirdiğin zip'leri backend `models/vit/` ve `models/swin/` klasörlerine aç. İçinde olacak:
- `model.onnx`
- `config.json`
- `preprocessor_config.json`

FastAPI tarafında:

```python
from transformers import AutoImageProcessor
import onnxruntime as ort, numpy as np
from PIL import Image

processor = AutoImageProcessor.from_pretrained('models/vit')
session = ort.InferenceSession('models/vit/model.onnx',
                                providers=['CPUExecutionProvider'])

def predict(image_bytes):
    img = Image.open(io.BytesIO(image_bytes)).convert('RGB')
    inputs = processor(images=img, return_tensors='np')
    logits = session.run(None, {'pixel_values': inputs['pixel_values'].astype(np.float32)})[0]
    probs = np.exp(logits) / np.exp(logits).sum(axis=1, keepdims=True)
    return {'is_fake': bool(probs[0,1] > 0.5), 'confidence': float(probs[0].max())}
```

Bu kadar.